# Chapter 03 — PDF, OCR and Layout (step by step)

*Where we are:* the **second parsing path**. Not every patent arrives as clean XML; many exist
only as **PDF**, a *presentation* format that says *where ink goes on a page*, not *what the
document means*.

```
[ PDF → text · words · numbers · tables · images ]  →  document model (+ bounding-box anchors)
[ scanned page → OCR → text · words · confidence  ]  →  same model, via recognition
```

**How this chapter works.** So that we always know the *ground truth*, we first **build our own
PDF** containing sentences, words, numbers, a table and an image. Then we extract **each construct
one at a time** with `pdfplumber` and `PyMuPDF`. Then we do the **OCR** path the same way
(rasterize → recognize), and finish on a **genuine patent PDF**.

Every implementation is written **inline and fully commented**; the same helpers are also packaged
in `patentrag/parsing.py` for the rest of the series to import.

In [1]:
# === Chapter 03 · standard bootstrap (identical pattern in every notebook) ===
# Runs standalone on a fresh Google Colab VM *or* a local checkout.
import os, sys, subprocess

REPO_URL = "https://github.com/rsalehin/patent-rag-masterclass"
NEED_OCR = True
IN_COLAB = "google.colab" in sys.modules


def _clone_repo(url, target):
    """Clone the repo on Colab. For a PRIVATE repo, authenticate with a GitHub token read from
    Colab Secrets (key 'GITHUB_TOKEN') or the GITHUB_TOKEN env var. The token is never printed."""
    token = None
    try:
        from google.colab import userdata  # type: ignore
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = os.environ.get("GITHUB_TOKEN")
    auth_url = url
    if token and url.startswith("https://github.com/"):
        auth_url = url.replace("https://github.com/", f"https://{token}@github.com/")
    r = subprocess.run(["git", "clone", "--depth", "1", auth_url, target],
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)  # avoid leaking the token
    if r.returncode != 0:
        raise RuntimeError(
            "git clone failed. This is a PRIVATE repo, so Colab needs a GitHub token:\n"
            "  1) Create a token (scope: repo) at https://github.com/settings/tokens\n"
            "  2) In Colab, open the key icon (Secrets) in the left sidebar, add a secret named\n"
            "     GITHUB_TOKEN, paste the token, and enable 'Notebook access'.\n"
            "  3) Re-run this cell.\n"
            "  (Alternatively, make the GitHub repo public — then no token is needed.)")


if IN_COLAB:
    target = "/content/patent-rag-masterclass"
    if not os.path.isdir(target):
        if not REPO_URL:
            raise RuntimeError("Set REPO_URL to this repo's GitHub URL (see README.md).")
        _clone_repo(REPO_URL, target)
    os.chdir(target)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    if NEED_OCR:
        subprocess.run(["apt-get", "install", "-y", "-q", "tesseract-ocr"], check=False)

# Ensure the repo root (containing patentrag/) is importable.
for _cand in [os.getcwd()] + [os.path.dirname(os.getcwd())]:
    if os.path.isdir(os.path.join(_cand, "patentrag")):
        if _cand not in sys.path:
            sys.path.insert(0, _cand)
        break

from patentrag import bootstrap as bs
bs.setup_environment(REPO_URL, need_ocr=NEED_OCR)
bs.set_seeds()
_env = bs.environment_report()
print("Chapter 03 bootstrap OK")
print("  Python", _env["python"], "| Colab:", _env["in_colab"], "| CPU cores:", _env["cpu_count"])
print("  torch", _env["torch"], "| CUDA:", _env["cuda_available"], "| tesseract:", _env["tesseract"])

Chapter 03 bootstrap OK
  Python 3.12.10 | Colab: False | CPU cores: 24
  torch 2.12.0.dev20260304+cu130 | CUDA: True | tesseract: True


## 0. Build a demo PDF we control

Before extracting, we author a PDF whose contents we know exactly — that way every extraction can
be checked against the truth. We use **reportlab** (`platypus`, its high-level layout engine).
First we render a small chart to a PNG so we have a real **image** to embed.

In [2]:
import os                                            # filesystem paths
import matplotlib                                    # plotting library
matplotlib.use("Agg")                                # non-interactive backend (write files, no GUI window)
import matplotlib.pyplot as plt                      # the pyplot interface

os.makedirs(bs.ARTIFACTS, exist_ok=True)             # artifacts/ is our scratch dir (gitignored, rebuilt on Colab)
img_path = str(bs.ARTIFACTS / "demo_fig.png")        # where the chart PNG will live

fig, ax = plt.subplots(figsize=(3, 1.6))             # a small figure (3in x 1.6in)
ax.bar(["A", "B", "C"], [3, 7, 5], color="#4C78A8")  # three bars — arbitrary demo data
ax.set_title("Figure 1: demo chart")                 # a title so we can recognize the image later
fig.savefig(img_path, dpi=100, bbox_inches="tight")  # rasterize the figure to a PNG on disk
plt.close(fig)                                        # free the figure (avoid inline display here)
print("wrote image:", img_path, os.path.getsize(img_path), "bytes")

wrote image: D:\Projects\AI-ML with Jupyter Notebook\Version 2\artifacts\demo_fig.png 5291 bytes


In [3]:
# Now assemble the PDF from "flowables" (paragraphs, tables, images) with reportlab.platypus.
from reportlab.lib.pagesizes import LETTER                       # US Letter page size (612 x 792 pt)
from reportlab.lib import colors                                 # named colors for table styling
from reportlab.lib.styles import getSampleStyleSheet             # ready-made paragraph styles (Title, BodyText...)
from reportlab.platypus import (SimpleDocTemplate, Paragraph,    # document + text flowables
                                Spacer, Table, TableStyle, Image) # spacing, table, table-style, image flowables

demo_pdf = str(bs.ARTIFACTS / "demo_document.pdf")               # output path for our demo PDF
styles = getSampleStyleSheet()                                   # grab the default stylesheet

# `story` is the ordered list of flowables reportlab lays out top-to-bottom onto pages.
story = [
    Paragraph("Quarterly Patent Report", styles["Title"]),                       # a TITLE (big, centered)
    Paragraph("This document contains sentences, words, and numbers. "           # a PARAGRAPH of prose:
              "Vector search retrieves relevant patents quickly, and "           #   two full sentences we
              "approximate nearest neighbor indexing keeps it fast.", styles["BodyText"]),
    Paragraph("Invoice 12345 total 678.90 USD across 42 filings in 2026.",       # a line rich in NUMBERS
              styles["BodyText"]),
    Spacer(1, 12),                                                               # 12pt vertical gap
    Table(                                                                       # a 3x3 TABLE (header + 2 rows)
        [["Patent", "Year", "Claims"],                                           #   row 0 = header
         ["US9081550B2", "2015", "30"],                                          #   row 1 = data
         ["US8930304B2", "2015", "17"]],                                         #   row 2 = data
        style=TableStyle([                                                       # visual rules so it's a real table:
            ("GRID", (0, 0), (-1, -1), 0.5, colors.black),                       #   draw thin borders on every cell
            ("BACKGROUND", (0, 0), (-1, 0), colors.lightgrey),                   #   shade the header row
        ])),
    Spacer(1, 12),                                                               # another gap
    Image(img_path, width=200, height=110),                                      # embed the chart PNG (an IMAGE)
]
SimpleDocTemplate(demo_pdf, pagesize=LETTER).build(story)                         # lay out `story` and write the PDF
print("built:", demo_pdf, os.path.getsize(demo_pdf), "bytes")

built: D:\Projects\AI-ML with Jupyter Notebook\Version 2\artifacts\demo_document.pdf 8573 bytes


## 1. Why PDF is hard (and which tool for which job)

A PDF stores **positioned glyphs**, not paragraphs. Reconstructing meaning means reading glyph
coordinates back into words, lines, tables and figures. We use two complementary libraries:

| Tool | Best at |
|---|---|
| **pdfplumber** | text, **words with boxes**, and **tables** (it reasons about lines/rules) |
| **PyMuPDF** (`fitz`) | fast text/words, **image extraction**, page rasterization |

## 2. Extract text (sentences)

`extract_text()` walks the page's characters in reading order and joins them with spaces/newlines.

In [4]:
import pdfplumber                                     # the text/table extraction library
import re                                             # regex, used below to split sentences / find numbers

with pdfplumber.open(demo_pdf) as pdf:                # open() parses the file; the `with` closes it after
    page = pdf.pages[0]                               # pages is 0-indexed; grab the first (only) page
    page_text = page.extract_text()                   # reconstruct reading-order text from glyph positions

print("--- raw extracted text ---")
print(page_text)

# Split the block into individual sentences: cut *after* . ! or ? that is followed by whitespace.
sentences = re.split(r"(?<=[.!?])\s+", page_text.replace("\n", " ").strip())
print("\n--- sentences ---")
for i, s in enumerate(sentences, 1):                  # enumerate from 1 for human-friendly numbering
    print(f"  {i}. {s}")

--- raw extracted text ---
Quarterly Patent Report
This document contains sentences, words, and numbers. Vector search retrieves relevant patents
quickly, and approximate nearest neighbor indexing keeps it fast.
Invoice 12345 total 678.90 USD across 42 filings in 2026.
Patent Year Claims
US9081550B2 2015 30
US8930304B2 2015 17

--- sentences ---
  1. Quarterly Patent Report This document contains sentences, words, and numbers.
  2. Vector search retrieves relevant patents quickly, and approximate nearest neighbor indexing keeps it fast.
  3. Invoice 12345 total 678.90 USD across 42 filings in 2026.
  4. Patent Year Claims US9081550B2 2015 30 US8930304B2 2015 17


## 3. Extract words + bounding boxes

`extract_words()` returns each word with its **bounding box**. pdfplumber's coordinates are
`x0`/`x1` (left/right) and `top`/`bottom` (distance **down from the top** of the page), in PDF
points (72 per inch). These boxes are what a citation eventually uses to **highlight** the source.

In [5]:
import pandas as pd                                   # tables for display

with pdfplumber.open(demo_pdf) as pdf:                # reopen (each `with` block is self-contained)
    words = pdf.pages[0].extract_words()              # list of dicts: {text, x0, x1, top, bottom, ...}

print("word count:", len(words))                      # how many words on the page
# Show the first few with just the fields we care about, rounded for readability.
words_df = pd.DataFrame(words)[["text", "x0", "x1", "top", "bottom"]].round(1)
words_df.head(8)

word count: 43


,text,x0,x1,top,bottom
0,Quarterly,204.5,284.5,81.7,99.7
1,Patent,289.5,344.5,81.7,99.7
2,Report,349.5,407.5,81.7,99.7
3,This,78.0,96.9,108.1,118.1
4,document,99.7,143.6,108.1,118.1
5,contains,146.4,183.6,108.1,118.1
6,"sentences,",186.4,234.7,108.1,118.1
7,"words,",237.5,267.0,108.1,118.1


## 4. Extract numbers

Numbers (quantities, dates, identifiers, prices) matter in patents. Once we have the text we find
them with a regex — a digit followed by more digits, dots or commas.

In [6]:
# \d      : a digit (0-9)
# [\d.,]* : then zero or more digits, dots, or commas (captures 678.90, 12,345, 2026)
numbers = re.findall(r"\d[\d.,]*", page_text)         # scan the whole page text for number-like tokens
print("numbers found:", numbers)
# We can also keep each number's location by filtering the word boxes to number-like words:
number_words = [w for w in words if re.fullmatch(r"\d[\d.,]*", w["text"])]
print("located numbers:", [(w["text"], round(w["x0"], 0), round(w["top"], 0)) for w in number_words])

numbers found: ['12345', '678.90', '42', '2026.', '9081550', '2', '2015', '30', '8930304', '2', '2015', '17']
located numbers: [('12345', 112.0, 138.0), ('678.90', 165.0, 138.0), ('42', 254.0, 138.0), ('2026.', 307.0, 138.0), ('2015', 312.0, 183.0), ('30', 346.0, 183.0), ('2015', 312.0, 201.0), ('17', 346.0, 201.0)]


## 5. Extract the table

Free text loses a table's structure. `extract_tables()` uses the ruling lines we drew to recover a
**list of rows**, each a list of cells — which we can load straight into a DataFrame.

In [7]:
with pdfplumber.open(demo_pdf) as pdf:                # reopen for the table pass
    tables = pdf.pages[0].extract_tables()            # returns a list of tables; each table is list-of-rows

print("tables found:", len(tables))                   # our page has exactly one
raw = tables[0]                                        # take the first table (rows of cells)
table_df = pd.DataFrame(raw[1:], columns=raw[0])       # row 0 is the header; the rest are data rows
table_df

tables found: 1


,Patent,Year,Claims
0,US9081550B2,2015,30
1,US8930304B2,2015,17


## 6. Extract images

Figures live as embedded image objects. `PyMuPDF` lists them per page (`get_images`) and pulls the
raw bytes back out (`extract_image`) — recovering the exact PNG we embedded.

In [8]:
import fitz                                            # PyMuPDF is imported as `fitz`

doc = fitz.open(demo_pdf)                              # open the PDF (a Document object)
page0 = doc[0]                                         # first page
image_list = page0.get_images(full=True)              # every image xref referenced by this page
print("images on page 0:", len(image_list))

xref = image_list[0][0]                                # the first image's cross-reference number
base = doc.extract_image(xref)                         # returns {'image': raw bytes, 'ext': 'png', ...}
out_img = str(bs.ARTIFACTS / f"extracted.{base['ext']}")
with open(out_img, "wb") as fh:                        # write the recovered image to disk
    fh.write(base["image"])
print("extracted:", out_img, "| format:", base["ext"], "| bytes:", len(base["image"]))
doc.close()                                            # release the file handle

# Display the recovered image to prove it round-tripped.
from PIL import Image as PILImage                       # Pillow to load the PNG
plt.figure(figsize=(3, 1.6)); plt.imshow(PILImage.open(out_img)); plt.axis("off")
plt.title("image recovered from the PDF"); plt.show()

images on page 0: 1
extracted: D:\Projects\AI-ML with Jupyter Notebook\Version 2\artifacts\extracted.png | format: png | bytes: 4917


C:\Users\rsalehin\AppData\Local\Temp\ipykernel_42244\3749273324.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.title("image recovered from the PDF"); plt.show()


## 7. Words + boxes with PyMuPDF (and draw them)

PyMuPDF's `get_text("words")` returns `(x0, y0, x1, y1, "word", block, line, word_no)` — here the
box is `(x0,y0)`=top-left, `(x1,y1)`=bottom-right, in points. We overlay the first boxes on a
rasterized page so the coordinates become tangible.

> This is the inline version of `patentrag.parsing.pdf_words()` — the packaged helper returns
> exactly these `(text, x0, y0, x1, y1)` tuples.

In [9]:
def pdf_words(path, page=0):
    """Return words with bounding boxes as (text, x0, y0, x1, y1) — mirrors patentrag.parsing."""
    with fitz.open(str(path)) as d:                    # open the document (closed automatically)
        # get_text("words") -> list of tuples; indices 0..3 are the box, 4 is the word string
        return [(w[4], w[0], w[1], w[2], w[3]) for w in d[page].get_text("words")]

pm_words = pdf_words(demo_pdf, 0)                       # extract via our helper
print("PyMuPDF words:", len(pm_words), "| first:", pm_words[0])

from matplotlib.patches import Rectangle               # to draw rectangles over the page image
with fitz.open(demo_pdf) as d:                          # render page 0 to a raster image at 150 DPI
    pix = d[0].get_pixmap(dpi=150)                      # a pixmap = RGB pixel buffer of the page
img = PILImage.frombytes("RGB", (pix.width, pix.height), pix.samples)  # wrap pixels as a PIL image
scale = 150 / 72.0                                      # PDF points -> pixels (image is 150 DPI, PDF is 72/in)

fig, ax = plt.subplots(figsize=(6, 7)); ax.imshow(img)  # show the page raster
for (t, x0, y0, x1, y1) in pm_words[:30]:               # overlay the first 30 word boxes
    ax.add_patch(Rectangle((x0*scale, y0*scale), (x1-x0)*scale, (y1-y0)*scale,   # scale points->pixels
                           fill=False, edgecolor="#E4572E", linewidth=0.7))
ax.set_title("first 30 word boxes"); ax.axis("off"); plt.show()

PyMuPDF words: 43 | first: ('Quarterly', 204.48001098632812, 76.73999786376953, 284.50799560546875, 101.5260009765625)


C:\Users\rsalehin\AppData\Local\Temp\ipykernel_42244\3539938265.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.set_title("first 30 word boxes"); ax.axis("off"); plt.show()


## 8. Bounding box → `SourceAnchor` (provenance)

A word box only helps if it plugs into the shared provenance types. We wrap it into a
`BoundingBox` + `SourceAnchor` (from `patentrag.models`, Chapter 01) so a PDF citation and an XML
citation speak the same language. This is the inline body of `patentrag.parsing.word_anchor()`.

In [10]:
from patentrag.models import BoundingBox, SourceAnchor  # the canonical provenance types (Ch 01)

def word_anchor(document_id, page, word_box):
    """Wrap a (text, x0, y0, x1, y1) word into a SourceAnchor carrying a BoundingBox."""
    _, x0, y0, x1, y1 = word_box                        # unpack; we ignore the text here
    return SourceAnchor(                                 # the provenance object a citation stores
        document_id=document_id,                         # which document this anchor belongs to
        page=page,                                       # 0-based page index
        bbox=BoundingBox(page=page, x0=x0, y0=y0, x1=x1, y1=y1),  # the exact rectangle to highlight
    )

anchor = word_anchor("demo_document", 0, pm_words[10])  # build an anchor for the 11th word
print("word:", repr(pm_words[10][0]))
print("anchor:", anchor.model_dump(exclude_none=True))  # dump non-null fields to see the structure

word: 'Vector'
anchor: {'document_id': 'demo_document', 'page': 0, 'bbox': {'page': 0, 'x0': 333.66998291015625, 'y0': 105.25, 'x1': 362.5699462890625, 'y1': 118.98999786376953}}


## 9. OCR — when there is no text layer

Everything so far used the PDF's **text layer** (born-digital). A **scanned** patent is just
pixels: you must **rasterize → recognize (OCR)**. We simulate a scan by rasterizing our demo page,
then OCR it with **Tesseract** via `pytesseract`, and compare against the known text.

First, rasterize (inline `patentrag.parsing.rasterize_page`).

In [11]:
def rasterize_page(path, page=0, dpi=150):
    """Render a PDF page to a PIL image at the given DPI (higher DPI = better OCR, slower)."""
    with fitz.open(str(path)) as d:                     # open doc
        pix = d[page].get_pixmap(dpi=dpi)               # rasterize the page to pixels
    return PILImage.frombytes("RGB", (pix.width, pix.height), pix.samples)  # wrap as PIL image

scan = rasterize_page(demo_pdf, 0, dpi=150)             # our "scanned page" (no text layer, just pixels)
print("rasterized page size (px):", scan.size)

rasterized page size (px): (1275, 1650)


### 9a. OCR text and per-word confidence

`image_to_string` returns the recognized text; `image_to_data` returns **per-word** results
including a **confidence** (0–100) and a pixel box. This is the inline body of
`patentrag.parsing.ocr_image()`.

In [12]:
import pytesseract                                      # Python wrapper around the Tesseract engine
from pytesseract import Output                          # to request dict-shaped output

cmd = bs.find_tesseract()                               # locate the tesseract binary (Windows path / PATH)
if cmd:                                                 # if found, point pytesseract at it
    pytesseract.pytesseract.tesseract_cmd = cmd

ocr_text = pytesseract.image_to_string(scan)            # full-page OCR -> one text string
data = pytesseract.image_to_data(scan, output_type=Output.DICT)  # per-word: text/conf/left/top/width/height

# Keep only real words with a valid confidence (Tesseract emits -1 for layout-only rows).
ocr_words = [{"text": t, "conf": float(c),
              "box": (data["left"][i], data["top"][i], data["width"][i], data["height"][i])}
             for i, (t, c) in enumerate(zip(data["text"], data["conf"])) if t.strip() and float(c) >= 0]
mean_conf = sum(w["conf"] for w in ocr_words) / len(ocr_words)   # average confidence over words

print("OCR text (head):", repr(ocr_text[:80]))
print("OCR words:", len(ocr_words), "| mean confidence: %.1f" % mean_conf)

OCR text (head): 'Quarterly Patent Report\n\nThis document contains sentences, words, and numbers. V'
OCR words: 54 | mean confidence: 93.9


### 9b. OCR reads the table and numbers too

OCR is content-agnostic — it transcribes whatever pixels look like text, including table cells and
numbers (though structure is lost; you'd re-detect the table from geometry).

In [13]:
ocr_tokens = [w["text"] for w in ocr_words]             # just the recognized strings
# Did OCR recover our table cells and numbers?
print("table cells seen:", [t for t in ocr_tokens if t in ("US9081550B2", "2015", "30", "17", "Patent")])
print("numbers seen     :", re.findall(r"\d[\d.,]*", ocr_text)[:8])
# The lowest-confidence words are where you would NOT trust the transcription:
low = sorted(ocr_words, key=lambda w: w["conf"])[:5]
pd.DataFrame([{"text": w["text"], "confidence": w["conf"]} for w in low])

table cells seen: ['Patent', 'Patent', 'US9081550B2', '2015', '30', '2015', '17']
numbers seen     : ['12345', '678.90', '42', '2026.', '9081550', '2', '2015', '30']


,text,confidence
0,SOO,17.0
1,oN,82.0
2,patents,87.0
3,US9081550B2,92.0
4,US8930304B2,92.0


### 9c. Text layer vs OCR — measure the gap

Even on a clean page, OCR ≠ the text layer. We quantify the difference with **character error
rate** (CER) = edit distance / reference length. We implement the classic **Levenshtein** dynamic
program inline (this is `patentrag.parsing.char_error_rate`).

In [14]:
def char_error_rate(reference, hypothesis):
    """CER = Levenshtein(reference, hypothesis) / len(reference), on whitespace-collapsed strings."""
    ref = re.sub(r"\s+", " ", reference).strip()        # normalize whitespace so spacing isn't "errors"
    hyp = re.sub(r"\s+", " ", hypothesis).strip()
    if not ref:                                          # guard: empty reference -> no error
        return 0.0
    m, n = len(ref), len(hyp)                            # sizes of the two strings
    prev = list(range(n + 1))                            # DP row 0: cost of turning "" into hyp[:j] = j
    for i in range(1, m + 1):                            # for each reference character...
        cur = [i] + [0] * n                              # column 0: cost of deleting i ref chars
        for j in range(1, n + 1):                        # for each hypothesis character...
            cost = 0 if ref[i - 1] == hyp[j - 1] else 1  # 0 if chars match, else 1 (substitution)
            cur[j] = min(prev[j] + 1,                    # deletion  (drop a ref char)
                         cur[j - 1] + 1,                 # insertion (add a hyp char)
                         prev[j - 1] + cost)             # match/substitute
        prev = cur                                       # this row becomes "previous" for the next i
    return prev[n] / m                                   # bottom-right cell = edit distance; normalize

# On our clean, single-column demo page OCR should be near-perfect (low CER):
with pdfplumber.open(demo_pdf) as pdf:
    demo_layer = pdf.pages[0].extract_text()            # the ground-truth text layer
cer_demo = char_error_rate(demo_layer, ocr_text)        # compare against OCR
print(f"demo page CER (text-layer vs OCR): {cer_demo:.3f}  (low = clean, single column)")

demo page CER (text-layer vs OCR): 0.126  (low = clean, single column)


## 10. The real thing — a genuine patent PDF

Now the same pipeline on a **genuine born-digital patent PDF** (bundled). Real patents are
**two-column**, so OCR's linear reading order diverges from the logical order — the CER is much
higher, which is the core lesson: **OCR output ≠ the semantic document.**

In [15]:
patent_pdf = sorted((bs.DATA / "pdf").glob("*.pdf"))[0]  # first bundled genuine patent PDF
with fitz.open(patent_pdf) as d:                         # inspect it
    n_pages = d.page_count                               # real patents span several pages
print("genuine patent PDF:", patent_pdf.name, "|", n_pages, "pages")

with pdfplumber.open(patent_pdf) as pdf:                 # extract page-0 words via pdfplumber
    p_words = pdf.pages[0].extract_words()
    p_text = pdf.pages[0].extract_text()
print("page-0 words:", len(p_words), "| text head:", repr(p_text[:70]))

p_scan = rasterize_page(patent_pdf, 0, dpi=150)          # rasterize page 0 (simulate scanning)
p_ocr = pytesseract.image_to_string(p_scan)              # OCR it
cer_patent = char_error_rate(p_text, p_ocr)              # compare text layer vs OCR
print(f"patent page CER: {cer_patent:.3f}  (higher — two-column reading order + header noise)")

genuine patent PDF: US10083169B1.pdf | 12 pages
page-0 words: 740 | text head: 'TOMTOM DU IN UNA IMANI\nUS010083169B1\n(1 2 ) United States Patent\n( 10)'


patent page CER: 0.271  (higher — two-column reading order + header noise)


## 11. These helpers are packaged in `patentrag/parsing.py`

Everything you wrote inline above is mirrored in the library so later chapters import it instead of
re-deriving it — the code is the same, just reused:

| Inline here | Packaged as |
|---|---|
| `pdf_words(...)` | `patentrag.parsing.pdf_words` |
| `rasterize_page(...)` | `patentrag.parsing.rasterize_page` |
| the `image_to_data` block | `patentrag.parsing.ocr_image` |
| `char_error_rate(...)` | `patentrag.parsing.char_error_rate` |
| `word_anchor(...)` | `patentrag.parsing.word_anchor` |
| `extract_text` / `pdf_page_text` | `patentrag.parsing.pdf_page_text` |

In [16]:
from patentrag import parsing as P                       # the packaged versions
# Prove the packaged helper agrees with our inline extraction (same word count on the demo page):
assert len(P.pdf_words(demo_pdf, 0)) == len(pm_words)
print("packaged patentrag.parsing.pdf_words matches the inline version:", True)

packaged patentrag.parsing.pdf_words matches the inline version: True


## 12. A lightweight layout signal

Full layout models (Docling, LayoutParser, PP-Structure, Surya) recover reading order and regions
at real compute cost. A cheap structural cue costs nothing: the histogram of word **x-positions**
reveals columns. Two peaks ⇒ two columns (why the patent's OCR reading order broke above).

In [17]:
import numpy as np                                       # for the histogram / median
xs = np.array([(w[1] + w[3]) / 2 for w in P.pdf_words(patent_pdf, 0)])  # x-center of each word (points)
fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(xs, bins=40, color="#4C78A8")                    # distribution of horizontal positions
ax.set_xlabel("word x-center (points)"); ax.set_ylabel("word count")
ax.set_title("Word x-position histogram → two peaks = two columns"); plt.show()
left = int((xs < np.median(xs)).sum()); right = int((xs >= np.median(xs)).sum())  # split at the median
print(f"words left of median x: {left} | right: {right} — consistent with a two-column patent page")

words left of median x: 398 | right: 399 — consistent with a two-column patent page


C:\Users\rsalehin\AppData\Local\Temp\ipykernel_42244\144269833.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.set_title("Word x-position histogram → two peaks = two columns"); plt.show()


**Production implications.** Prefer the born-digital text layer when present (fast, exact); fall
back to OCR for scans and **carry confidence downstream** so weak spans can be de-weighted or
re-OCR'd. Keep every word's bounding box — it is the anchor a citation highlights. Tables need
geometry-aware extraction (pdfplumber) or a layout model; images need object extraction (PyMuPDF).

## Chapter invariants

In [18]:
# The demo page round-trips: our known content is recovered by each extractor.
assert "Quarterly Patent Report" in page_text                     # text layer
assert len(words) >= 20 and all("x0" in w for w in words)          # words + boxes
assert "12345" in numbers and "678.90" in numbers                  # numbers
assert table_df.shape == (2, 3) and list(table_df.columns) == ["Patent", "Year", "Claims"]  # table
assert base["ext"] == "png" and len(base["image"]) > 100           # image extraction
assert anchor.bbox is not None and anchor.bbox.page == 0           # bbox -> SourceAnchor
assert P.ocr_available() and mean_conf > 0                         # OCR ran with real confidence
assert cer_demo <= cer_patent                                      # clean page easier than 2-column patent
assert char_error_rate("abc", "abc") == 0.0                        # CER sanity: identical -> 0
print("All Chapter 03 invariants hold.")

All Chapter 03 invariants hold.


In [19]:
# === Chapter 03 validation footer ===
import time, platform, sys, importlib.metadata as _md
_pkgs = ['pdfplumber', 'PyMuPDF', 'pytesseract', 'reportlab', 'pillow', 'matplotlib']
print("Chapter 03 — environment")
print("  Python :", sys.version.split()[0], "on", platform.system(), platform.release())
for _p in _pkgs:
    try: print(f"  {_p:24}: {_md.version(_p)}")
    except Exception: print(f"  {_p:24}: (not installed)")
print()
print("CHAPTER 03 VALIDATION: PASS")

Chapter 03 — environment
  Python : 3.12.10 on Windows 11
  pdfplumber              : 0.11.10
  PyMuPDF                 : 1.27.2.3
  pytesseract             : 0.3.13
  reportlab               : 5.0.1
  pillow                  : 12.3.0
  matplotlib              : 3.11.1

CHAPTER 03 VALIDATION: PASS
